In [75]:
from importlib.metadata import version
import tiktoken
import torch
print("PyTorch version:", torch.__version__)
print("tiktoken版本:", version("tiktoken"))
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

PyTorch version: 2.10.0+cpu
tiktoken版本: 0.12.0
torch version: 2.10.0
tiktoken version: 0.12.0


In [2]:
import re
text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [3]:
with open("第一章-笼中雀.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])
print()

pattern = r'([，。！？；：,.!?;:\s]|--|\(|\)|“|”|‘|’|\"|\')'

# 原始分割
preprocessed = re.split(pattern, raw_text)
# 去掉空白、空字符串
preprocessed1 = [item.strip() for item in preprocessed if item.strip()]
# 中文连续文字拆成单字
final_tokens = []
for t in preprocessed1:
    if re.match(r'^[\u4e00-\u9fff]+$', t):  
        final_tokens.extend(list(t))
    else:
        final_tokens.append(t)
# 打印前50个 token
print(final_tokens[:50])
print(len(final_tokens))

Total number of character: 6317
第1章 惊蛰
二月二，龙抬头。

暮色里，小镇名叫泥瓶巷的僻静地方，有位孤苦伶仃的清瘦少年，此时他正按照习俗，一手持蜡烛，一手持桃枝，照耀房梁、墙壁、木床等处，用桃枝敲敲打打，试图借此驱赶蛇蝎、蜈蚣

['第1章', '惊', '蛰', '二', '月', '二', '，', '龙', '抬', '头', '。', '暮', '色', '里', '，', '小', '镇', '名', '叫', '泥', '瓶', '巷', '的', '僻', '静', '地', '方', '，', '有', '位', '孤', '苦', '伶', '仃', '的', '清', '瘦', '少', '年', '，', '此', '时', '他', '正', '按', '照', '习', '俗', '，', '一']
6033


re.match(r'^[\u4e00-\u9fff]+$', t)
✅ 只匹配 全是中文汉字的字符串
\u4e00-\u9fff 是 Unicode 中文汉字范围
一旦 token 里有非汉字字符（数字、英文、符号），match 就不成立
比如 "第1章" 中有数字 '1' → 不满足 ^[\u4e00-\u9fff]+$
所以整个 "第1章" 被当作一个 token 直接保留下来，没有拆开

这一步是创建一个词汇表

In [25]:
all_words = sorted(set(final_tokens))
all_words.extend(["<|endoftext|>", "<|unk|>"])
vocab_size = len(all_words)
print(vocab_size) 

vocab = {token: i for i, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)


1148
('“', 0)
('”', 1)
('。', 2)
('一', 3)
('七', 4)
('万', 5)
('丈', 6)
('三', 7)
('上', 8)
('下', 9)
('不', 10)
('与', 11)
('丑', 12)
('且', 13)
('世', 14)
('东', 15)
('丢', 16)
('两', 17)
('严', 18)
('个', 19)
('丫', 20)
('中', 21)
('串', 22)
('为', 23)
('主', 24)
('么', 25)
('义', 26)
('之', 27)
('乎', 28)
('乏', 29)
('乐', 30)
('乘', 31)
('也', 32)
('习', 33)
('乡', 34)
('书', 35)
('买', 36)
('乱', 37)
('了', 38)
('事', 39)
('二', 40)
('于', 41)
('亏', 42)
('云', 43)
('五', 44)
('井', 45)
('些', 46)
('交', 47)
('京', 48)
('亮', 49)
('亲', 50)
('，', 1143)
('：', 1144)
('？', 1145)
('<|endoftext|>', 1146)
('<|unk|>', 1147)


现在将所有内容整合到一个分词器类中，这是针对中文文本的改版。

In [5]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        """
        中文单字、英文单词/数字、标点拆分
        """
        # 先用正则把标点和空白拆出来
        pattern = r'([，。！？；：,.!?;:\s]|--|\(|\)|“|”|‘|’|\"|\')'
        tokens = re.split(pattern, text)
        tokens = [t.strip() for t in tokens if t.strip()]
        final_tokens = []
        for t in tokens:
            # 拆中文汉字、数字、英文单词、标点
            chars = re.findall(r'[\u4e00-\u9fff]|[0-9]+|[a-zA-Z]+|[^\w\s]', t)

        # 转为 id
        ids = [self.str_to_int[t] for t in final_tokens]
        return ids

    def decode(self, ids):
        """
        中文不加空格，英文单词之间保留空格，标点前不加空格
        """
        tokens = [self.int_to_str[i] for i in ids]
        text = ""
        for i, token in enumerate(tokens):
            if i == 0:
                text += token
                continue
            # 标点前不加空格
            if re.match(r'[，。！？；：,.!?;:"\'“”‘’()\-]', token):
                text += token
            # 前一个是标点，不加空格
            elif re.match(r'[，。！？；：,.!?;:"\'“”‘’()\-]', tokens[i-1]):
                text += token
            # 英文单词/数字之间加空格
            elif re.match(r'^[a-zA-Z0-9]+$', token) and re.match(r'^[a-zA-Z0-9]+$', tokens[i-1]):
                text += " " + token
            # 中文字符直接拼接
            else:
                text += token
        return text

In [19]:
tokenizer = SimpleTokenizerV1(vocab)
text = """不过如姚老头这般钻牛角尖的人，终究少数。"""
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[10, 1014, 296, 301, 854, 286, 1019, 886, 1054, 688, 941, 340, 730, 51, 1143, 833, 785, 339, 529, 2]
不过如姚老头这般钻牛角尖的人，终究少数。


# 如果是是词汇里没有的词，将导致以下错误：
KeyError: ‘Hello
所以，
我们需要修改分词器以处理未知词汇。我们还需要解决特殊上下文标记的使用和
添加，这些标记可以增强模型对文本上下文或其他相关信息的理解。

In [26]:
import re

class SimpleTokenizerV2:
    def __init__(self, vocab, unk_token="<|unk|>"):
        """
        vocab: dict, token -> id
        unk_token: str, 用于未登录 token
        """
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}
        self.unk_token = unk_token
        if unk_token not in self.str_to_int:
            raise ValueError(f"{unk_token} must be in vocab")

    def encode(self, text):
        """
        中文单字 + 英文单词/数字 + 标点拆分
        未登录 token 映射到 <|unk|>
        """
        # 先用正则把标点和空白拆出来
        pattern = r'([，。！？；：,.!?;:\s]|--|\(|\)|“|”|‘|’|\"|\')'
        tokens = re.split(pattern, text)
        tokens = [t.strip() for t in tokens if t.strip()]

        final_tokens = []
        for t in tokens:
            # 拆中文汉字、数字、英文单词、标点
            chars = re.findall(r'[\u4e00-\u9fff]|[0-9]+|[a-zA-Z]+|[^\w\s]', t)
            final_tokens.extend(chars)

        # 转为 id，未登录 token 用 <|unk|>
        ids = [self.str_to_int.get(t, self.str_to_int[self.unk_token]) for t in final_tokens]
        return ids

    def decode(self, ids):
        """
        中文不加空格，英文单词/数字之间保留空格，标点前不加空格
        """
        tokens = [self.int_to_str.get(i, self.unk_token) for i in ids]
        text = ""
        for i, token in enumerate(tokens):
            if i == 0:
                text += token
                continue
            # 标点前不加空格
            if re.match(r'[，。！？；：,.!?;:"\'“”‘’()\-]', token):
                text += token
            # 前一个是标点，不加空格
            elif re.match(r'[，。！？；：,.!?;:"\'“”‘’()\-]', tokens[i-1]):
                text += token
            # 英文单词/数字之间加空格
            elif re.match(r'^[a-zA-Z0-9]+$', token) and re.match(r'^[a-zA-Z0-9]+$', tokens[i-1]):
                text += " " + token
            # 中文字符直接拼接
            else:
                text += token
        return text

In [29]:
tokenizer = SimpleTokenizerV2(vocab)
text = "不过如姚老头这般钻牛角尖的人，终究少数。Hello 世界！"
ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[10, 1014, 296, 301, 854, 286, 1019, 886, 1054, 688, 941, 340, 730, 51, 1143, 833, 785, 339, 529, 2, 1147, 14, 1147, 1142]
不过如姚老头这般钻牛角尖的人，终究少数。<|unk|>世<|unk|>！


# 2.5 BytePair encoding  2.5 BytePair 编码
GPT-2 使用字节对编码（BPE）作为其分词器
• 它允许模型将不在其预定义词汇表中的单词分解为更小的子词单元，甚至单个字符，从而能够处理词汇表外的单词
• 例如，如果 GPT-2 的词汇表中没有“unfamiliarword”这个词，它可能会将其标记unfam", "iliar", "word"] 或其他子词分解，具体取决于其训练的 BPE 合并规则
• 原始的 BPE 分词器可在此处找到：https://github.com/openai/gpt-2/blob/master/src/encoder.py
• 在本章中，我们使用的是 OpenAI 开源的 tiktoken 库中的 BPE 分词器，该库用 Rust 实现其核心算法以提高计算性能
• 我在 ./bytepair_encoder 中创建了一个笔记本，对这两种实现进行了并排比较（在示例文本上，tiktoken 的速度大约快 5 倍）

In [51]:
text = ("泥瓶巷家家户户的黄土院墙都很低矮，"
        "其实邻居少年完全不用踮起脚跟，"
        "可每次跟陈平安说话，偏偏喜欢蹲在墙头上。")
tokenizer = tiktoken.get_encoding("gpt2")
integers = tokenizer.encode(text, allowed_special={""})
print(integers)
strings = tokenizer.decode(integers)
print(strings)

[37345, 98, 163, 241, 114, 32432, 115, 22522, 114, 22522, 114, 22755, 115, 22755, 115, 21410, 165, 119, 226, 28839, 253, 165, 247, 95, 161, 95, 247, 32849, 121, 36181, 230, 19526, 236, 163, 253, 106, 171, 120, 234, 17739, 114, 22522, 252, 165, 224, 119, 161, 109, 227, 22887, 239, 33176, 112, 22522, 234, 17739, 101, 38834, 18796, 101, 164, 116, 106, 164, 113, 115, 164, 226, 248, 164, 115, 253, 171, 120, 234, 20998, 107, 162, 107, 237, 162, 105, 94, 164, 115, 253, 165, 247, 230, 33176, 111, 22522, 231, 46237, 112, 46237, 251, 171, 120, 234, 161, 223, 237, 161, 223, 237, 161, 244, 250, 162, 105, 95, 164, 117, 110, 28839, 101, 161, 95, 247, 13783, 112, 41468, 16764]
泥瓶巷家家户户的黄土院墙都很低矮，其实邻居少年完全不用踮起脚跟，可每次跟陈平安说话，偏偏喜欢蹲在墙头上。


In [52]:
with open("第一章-笼中雀.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

13736


In [57]:
enc_sample = enc_text[50:] # 从第50个开始，移出50个单词。

In [66]:
context_size = 10 # 用 4 个 token 作为上下文

x = enc_sample[:context_size] # 输入序列（context）
y = enc_sample[1:context_size+1] # 输入序列（context），就是 x 往右偏移一位，训练模型预测下一个 token

print(f"x: {x}") 
print(f"y:      {y}")

x: [20998, 104, 37345, 98, 163, 241, 114, 32432, 115, 21410]
y:      [104, 37345, 98, 163, 241, 114, 32432, 115, 21410, 161]


In [67]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[20998] ----> 104
[20998, 104] ----> 37345
[20998, 104, 37345] ----> 98
[20998, 104, 37345, 98] ----> 163
[20998, 104, 37345, 98, 163] ----> 241
[20998, 104, 37345, 98, 163, 241] ----> 114
[20998, 104, 37345, 98, 163, 241, 114] ----> 32432
[20998, 104, 37345, 98, 163, 241, 114, 32432] ----> 115
[20998, 104, 37345, 98, 163, 241, 114, 32432, 115] ----> 21410
[20998, 104, 37345, 98, 163, 241, 114, 32432, 115, 21410] ----> 161


In [68]:

for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

� ----> �
叫 ----> �
叫� ----> �
叫泥 ----> �
叫泥� ----> �
叫泥� ----> �
叫泥瓶 ----> �
叫泥瓶� ----> �
叫泥瓶巷 ----> 的
叫泥瓶巷的 ----> �


In [77]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [78]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Create dataloader
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [81]:
with open("第一章-笼中雀.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [89]:

dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)
second_batch = next(data_iter)
print(second_batch)

[tensor([[163, 105, 105,  16]]), tensor([[  105,   105,    16, 44165]])]
[tensor([[  105,   105,    16, 44165]]), tensor([[  105,    16, 44165,   254]])]


请注意，我们在这里增加步长，以避免批次之间出现重叠，因为更多的重叠可能会导致过拟合增加

In [91]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[  163,   105,   105,    16],
        [44165,   254, 10545,   225],
        [  232,   164,   249,   108],
        [  198, 12859,   234, 17312],
        [  230, 12859,   234,   171],
        [  120,   234, 11737,   247],
        [  162,   232,   105, 13783],
        [  112, 16764,   198,   198]])

Targets:
 tensor([[  105,   105,    16, 44165],
        [  254, 10545,   225,   232],
        [  164,   249,   108,   198],
        [12859,   234, 17312,   230],
        [12859,   234,   171,   120],
        [  234, 11737,   247,   162],
        [  232,   105, 13783,   112],
        [16764,   198,   198,   162]])


In [103]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123) # 固定随机数生成器的起点
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)
print(embedding_layer(torch.tensor([3])))
print(embedding_layer(input_ids))

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


In [98]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)


Token IDs:
 tensor([[  163,   105,   105,    16],
        [44165,   254, 10545,   225],
        [  232,   164,   249,   108],
        [  198, 12859,   234, 17312],
        [  230, 12859,   234,   171],
        [  120,   234, 11737,   247],
        [  162,   232,   105, 13783],
        [  112, 16764,   198,   198]])

Inputs shape:
 torch.Size([8, 4])


In [99]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


In [101]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])


In [102]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])
